# Quality Assessment Method Comparison

Compare image quality assessment methods on PhotoTriage dataset:
- **Performance metrics**: Top-1/2 Accuracy, MRR, Runtime
- **Correlation analysis**: Which methods agree on series difficulty?
- **Method wins**: Cases where one method succeeds but others fail
- **Failure analysis**: Patterns in failure cases

## Configuration Options

**Option 1 - Single folder**: Specify exact benchmark directory
```python
AUTO_SCAN = False
BENCHMARK_DIR = "outputs/quality_benchmarks/benchmark_2025-11-14_00-06-59"
```

**Option 2 - Auto-scan** (recommended): Automatically find latest benchmark
```python
AUTO_SCAN = True
BENCHMARK_BASE_DIR = "outputs/quality_benchmarks"
```



In [ ]:
# === Imports ===
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from sim_bench.quality_assessment.analysis import (
    load_quality_results,
    merge_per_series_metrics,
    compute_correlation_matrix,
    plot_performance_comparison,
    plot_runtime_comparison,
    plot_efficiency_comparison,
    plot_correlation_heatmap,
    find_method_wins,
    visualize_method_wins,
    get_top_methods_by_accuracy,
    analyze_failures,
    plot_failure_analysis
)
from sim_bench.analysis.export import export_notebook_to_pdf, archive_notebook
from sim_bench.analysis.utils import get_project_root

sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

print("✓ Imports successful")



## Configuration



In [ ]:
# === Configuration ===
PROJECT_ROOT = get_project_root()
EXCLUDE_METHODS_REGEX = 'composite'
# Choose scanning mode
AUTO_SCAN = True  # Set to False to use specific folder

if AUTO_SCAN:
    # Option 1: Auto-scan from base directory (uses latest)
    BENCHMARK_BASE_DIR = PROJECT_ROOT / "outputs" / "pairwise_benchmark_3hour"
    BENCHMARK_DIR = None
    USE_LATEST = False
else:
    # Option 2: Specify exact benchmark directory
    BENCHMARK_BASE_DIR = None
    BENCHMARK_DIR = PROJECT_ROOT / "outputs" / "quality_benchmarks" / "benchmark_2025-11-14_00-06-59"
    USE_LATEST = False

# Analysis parameters
TOP_N_METHODS = 10  # Number of top methods to show in wins visualization
TOP_N_IMAGES = 3   # Number of top images to show per method

# Export options
EXPORT_HTML = True  # Export notebook to HTML at the end

print(f"Scanning mode: {'AUTO' if AUTO_SCAN else 'MANUAL'}")
if AUTO_SCAN:
    print(f"Base directory: {BENCHMARK_BASE_DIR.relative_to(PROJECT_ROOT)}")
else:
    print(f"Benchmark directory: {BENCHMARK_DIR.relative_to(PROJECT_ROOT)}")



## 1. Load Data



In [ ]:
# Load benchmark results
methods_summary_df, detailed_results_df, series_data_dict, benchmark_dir = load_quality_results(
    benchmark_dir=BENCHMARK_DIR,
    auto_scan=AUTO_SCAN,
    base_dir=BENCHMARK_BASE_DIR,
    use_latest=USE_LATEST
)

print(f"\n{'='*80}")
print("DATA SUMMARY")
print(f"{'='*80}")
print(f"Benchmark directory: {benchmark_dir.name}")
print(f"Methods found: {len(methods_summary_df)}")
print(f"Methods: {', '.join(methods_summary_df['method'].tolist())}")
print(f"\nMethods Summary:")
display(methods_summary_df)

print(f"\nDetailed Results:")
display(detailed_results_df)



## 2. Performance Metrics Comparison



In [ ]:
# Plot performance comparison
fig = plot_performance_comparison(methods_summary_df)
plt.show()



In [ ]:
# Plot runtime comparison
fig = plot_runtime_comparison(detailed_results_df)
plt.show()



In [ ]:
# Plot efficiency (accuracy vs speed)
fig = plot_efficiency_comparison(methods_summary_df)
plt.show()



## 3. Correlation Matrix

Analyze which methods agree on series difficulty (Top-1 Accuracy correlation).



In [ ]:
# Merge per-series data for correlation analysis
methods = methods_summary_df['method'].tolist()
merged_df = merge_per_series_metrics(series_data_dict, methods=methods, metric='correct')

print(f"Merged {len(merged_df)} series across {len(methods)} methods")

# Compute correlation matrix
corr_matrix = compute_correlation_matrix(merged_df, methods=methods, metric='correct')

print(f"\nCorrelation Matrix:")
display(corr_matrix)

# Plot correlation heatmap
fig = plot_correlation_heatmap(corr_matrix, methods=methods)
plt.show()



In [ ]:
df_clip = pd.read_csv(r'D:\sim-bench\outputs\pairwise_benchmark_3hour\pairwise_20251120_100520\CLIP-Aesthetic-LAION\per_pair_results.csv')
df_clip.head()

In [ ]:

df_expanded.head()




In [ ]:
df_expanded.query("compareID1 == 1 and compareID2 == 2")

In [ ]:
import json
df_example = pd.read_json(r'D:\Similar Images\automatic_triage_photo_series\train_val\reviews_trainval\reviews_trainval\000001.json')
df_expanded = pd.json_normalize(df_example['reviews'])


df_count = df_expanded.groupby(['compareID1', 'compareID2', 'userChoice']).size().reset_index(name='count')
df_count_pivoted = pd.pivot_table(df_count, values='count', index=['compareID1', 'compareID2'], 
                                columns=['userChoice'], aggfunc='sum').reset_index()
for col in ['LEFT', 'RIGHT']:
    if col not in df_count_pivoted.columns:
        df_count_pivoted[col] = 0
    df_count_pivoted[col] = df_count_pivoted[col].fillna(0)
df_count_pivoted['max_count'] = df_count_pivoted[['LEFT', 'RIGHT']].max(axis=1)


df_count_pivoted['MaxVote'] = df_count_pivoted['compareID2']
df_count_pivoted.loc[df_count_pivoted['LEFT'] > df_count_pivoted['RIGHT'], 'MaxVote'] = \
     df_count_pivoted.loc[df_count_pivoted['LEFT'] > df_count_pivoted['RIGHT'], 'compareID1']
df_count_pivoted['Total'] = df_count_pivoted['LEFT'] + df_count_pivoted['RIGHT']
df_count_pivoted['Agreement'] = (df_count_pivoted['max_count']) / df_count_pivoted['Total']
df_count_pivoted.head()












In [ ]:
df_rates = pd.read_csv(r'D:\Similar Images\automatic_triage_photo_series\train_val\labels_db.csv')
df_rates

In [ ]:
df_rates.query("(majority_label == 'no_specific_attribute')")[['series_id', 'compareID1', 'compareID2', 'LEFT', 'RIGHT', 'Total',
       'max_count', 'Agreement', 'majority_label']]

In [ ]:
df_labels = pd.read_csv(r'D:\Similar Images\automatic_triage_photo_series\train_val\reviews_trainval\photo_triage_pairs_keyword_labels.csv')


In [ ]:
display(df_labels[['series_id', 'compareID1', 'compareID2', 'LEFT', 'RIGHT', 'Total',
       'max_count', 'Agreement', 'MaxVote', 'num_reviewers', 'majority_label',]].head())
print(df_labels.columns)

In [ ]:
df_labels_embedding = pd.read_csv(r'D:\Similar Images\automatic_triage_photo_series\train_val\reviews_trainval\reviews_trainval\test\photo_triage_pairs_embedding_labels.csv')
display(df_labels_embedding[['series_id', 'compareID1', 'compareID2', 'LEFT', 'RIGHT', 'Total',
       'max_count', 'Agreement', 'MaxVote', 'num_reviewers', 'majority_label', ]].head())
print(df_labels_embedding.columns)

In [ ]:
df_rates.query("(majority_label == 'no_reason_given')")[['series_id', 'compareID1', 'compareID2', 'LEFT', 'RIGHT', 'Total',
       'max_count', 'Agreement', 'majority_label']]

## 4. Method Wins Visualization

Show cases where one method succeeds but others fail. Displays:
- Ground truth best image
- Top 3 scoring images for each method (with scores and filenames)



In [ ]:
# Get top N methods by accuracy
methods_summary_df_to_show = methods_summary_df.copy()
methods_summary_df_to_show = methods_summary_df_to_show[~methods_summary_df_to_show['method'].str.contains(EXCLUDE_METHODS_REGEX)]
top_methods = get_top_methods_by_accuracy(methods_summary_df_to_show, top_n=TOP_N_METHODS)
print(f"Top {TOP_N_METHODS} methods: {', '.join(top_methods)}")

# Find method wins (cases where one method succeeds but others fail)
wins_dict = find_method_wins(merged_df, methods=top_methods, top_n_series=3)

print(f"\nMethod Wins Found:")
for method, wins_df in wins_dict.items():
    print(f"  {method}: {len(wins_df)} winning series")



In [ ]:
# Visualize method wins
# Note: This will display images, so make sure image paths are accessible
output_dir = benchmark_dir / "analysis_reports" / "method_wins"
output_dir.mkdir(parents=True, exist_ok=True)

saved_figs = visualize_method_wins(
    wins_dict=wins_dict,
    series_data_dict=series_data_dict,
    top_n_methods=TOP_N_METHODS,
    top_n_images=TOP_N_IMAGES,
    output_dir=output_dir,
    show_plots=True
)

print(f"\n✓ Saved {len(saved_figs)} visualization figures to {output_dir}")



## 5. Failure Analysis

Analyze patterns in failure cases.



In [ ]:
# Analyze failures
failure_stats = analyze_failures(merged_df, series_data_dict, methods=methods)

print("Failure Statistics:")
print(f"  All methods fail: {failure_stats.get('all_fail_count', 0)} series")
print(f"\n  Single method success:")
for method, data in failure_stats.get('single_success', {}).items():
    print(f"    {method}: {data['count']} series")

# Plot failure analysis
fig = plot_failure_analysis(failure_stats, methods=methods)
plt.show()



## 6. Export to HTML

Export this notebook to HTML for easy sharing.



In [ ]:
# Export notebook to HTML
if EXPORT_HTML:
    # Get notebook path (works in Jupyter)
    try:
        import ipynbname
        notebook_path = ipynbname.path()
    except:
        # Fallback: assume we're in notebooks directory
        notebook_path = PROJECT_ROOT / "notebooks" / "quality_assessment_analysis.ipynb"
    
    output_dir = benchmark_dir / "analysis_reports"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        html_path = export_notebook_to_pdf(
            str(notebook_path),
            output_dir=str(output_dir),
            timestamp=True,
            prefix="quality_assessment_analysis"
        )
        print(f"✓ Exported to HTML: {html_path.name}")
        print(f"📁 Location: {html_path}")
    except Exception as e:
        print(f"⚠ Export failed: {e}")
        print("   Make sure nbconvert is installed: pip install nbconvert")
        print("   Or manually export: File -> Download as -> HTML")
else:
    print("HTML export disabled. Set EXPORT_HTML = True to enable.")

